# RoadFlood-VLM Extended Metrics Only


In [ ]:

from pathlib import Path
import os
def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root."
    )
PROJECT_ROOT = find_project_root()


In [ ]:

# EXTENDED TEXT-GENERATION AND GROUNDING METRICS
from __future__ import annotations
import json
import math
import re
from collections import Counter
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import torch
from bert_score import BERTScorer
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from rouge_score import rouge_scorer
def locate_predictions_csv() -> Path:
    candidate = globals().get("predictions_csv")
    if candidate is not None:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    run_dir = globals().get("RUN_DIR")
    if run_dir is not None:
        candidate = Path(run_dir) / "test_predictions.csv"
        if candidate.exists():
            return candidate
    evaluation_root = (
        PROJECT_ROOT
        / "outputs"
        / "roadflood_vlm_evaluation"
    )
    runs = sorted(evaluation_root.glob("evaluation_*"))
    for run in reversed(runs):
        candidate = run / "test_predictions.csv"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate test_predictions.csv."
    )
predictions_path = locate_predictions_csv()
metric_run_dir = predictions_path.parent
metric_df = pd.read_csv(predictions_path)
required_columns = {
    "reference",
    "prediction",
}
missing_columns = required_columns - set(metric_df.columns)
if missing_columns:
    raise KeyError(
        f"Missing required prediction columns: {missing_columns}"
    )
references = (
    metric_df["reference"]
    .fillna("")
    .astype(str)
    .tolist()
)
predictions = (
    metric_df["prediction"]
    .fillna("")
    .astype(str)
    .tolist()
)
# -------------------------------------------------------------------
# Standard text-generation metrics
# -------------------------------------------------------------------
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)
rouge_rows = []
for reference, prediction in zip(references, predictions):
    values = rouge.score(reference, prediction)
    rouge_rows.append(
        {
            "rouge1_precision": values["rouge1"].precision,
            "rouge1_recall": values["rouge1"].recall,
            "rouge1_f1": values["rouge1"].fmeasure,
            "rouge2_precision": values["rouge2"].precision,
            "rouge2_recall": values["rouge2"].recall,
            "rouge2_f1": values["rouge2"].fmeasure,
            "rougeL_precision": values["rougeL"].precision,
            "rougeL_recall": values["rougeL"].recall,
            "rougeL_f1": values["rougeL"].fmeasure,
        }
    )
rouge_df = pd.DataFrame(rouge_rows)
for column in rouge_df.columns:
    metric_df[column] = rouge_df[column]
def simple_tokens(text: str) -> list[str]:
    return re.findall(
        r"\b\w+(?:[.-]\w+)*\b",
        str(text).lower(),
    )
reference_tokens = [
    simple_tokens(text)
    for text in references
]
prediction_tokens = [
    simple_tokens(text)
    for text in predictions
]
smoothing = SmoothingFunction().method1
bleu = corpus_bleu(
    [[tokens] for tokens in reference_tokens],
    prediction_tokens,
    smoothing_function=smoothing,
)# -------------------------------------------------------------------
# BERTScore
# -------------------------------------------------------------------
bertscore_model = (
    os.environ.get(
        "ROADFLOOD_BERTSCORE_MODEL",
        "microsoft/deberta-xlarge-mnli",
    )
    if "os" in globals()
    else "microsoft/deberta-xlarge-mnli"
)
bertscore_batch_size = int(
    os.environ.get(
        "ROADFLOOD_BERTSCORE_BATCH_SIZE",
        "4",
    )
    if "os" in globals()
    else 4
)
bert_device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
bertscorer = BERTScorer(
    model_type=bertscore_model,
    device=bert_device,
    batch_size=bertscore_batch_size,
    rescale_with_baseline=False,
)
# Some recent Transformers versions assign an extremely large sentinel
# value to model_max_length, which causes an integer overflow in bert-score.
bertscorer._tokenizer.model_max_length = 512
bert_precision, bert_recall, bert_f1 = bertscorer.score(
    predictions,
    references,
    verbose=True,
)
metric_df["bertscore_precision"] = (
    bert_precision.cpu().numpy()
)
metric_df["bertscore_recall"] = (
    bert_recall.cpu().numpy()
)
metric_df["bertscore_f1"] = (
    bert_f1.cpu().numpy()
)
# -------------------------------------------------------------------
# Grounding-specific metrics
# -------------------------------------------------------------------
NUMBER_PATTERN = re.compile(
    r"(?<![\w.])-?\d+(?:,\d{3})*(?:\.\d+)?%?"
)
SCENE_ID_PATTERN = re.compile(
    r'"scene_id"\s*:\s*"([^"]+)"',
    flags=re.IGNORECASE,
)
FLOOD_BURDEN_PATTERNS = [
    re.compile(
        r'"scene_flood_burden"\s*:\s*"'
        r'(None|Low|Moderate|High)"',
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"(?:scene has a|identifies a|roadway flood burden is)"
        r"\s+(None|Low|Moderate|High)"
        r"\s+(?:roadway )?flood burden",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"roadway flood burden(?: class)? is"
        r"\s+(None|Low|Moderate|High)",
        flags=re.IGNORECASE,
    ),
]
DISRUPTION_PATTERNS = [
    re.compile(
        r'"critical_network_disruption"\s*:\s*"'
        r'(None|Low|Moderate|High)"',
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"critical-network disruption"
        r"(?: category| class)? is"
        r"\s+(None|Low|Moderate|High)",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"(None|Low|Moderate|High)"
        r"\s+critical-network disruption",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"(None|Low|Moderate|High)"
        r"\s+disruption (?:label|category|class)",
        flags=re.IGNORECASE,
    ),
]
def normalize_number(token: str) -> str:
    token = token.replace(",", "").strip()
    is_percent = token.endswith("%")
    if is_percent:
        token = token[:-1]
    try:
        number = float(token)
        if math.isclose(number, round(number)):
            normalized = str(int(round(number)))
        else:
            normalized = (
                f"{number:.6f}"
                .rstrip("0")
                .rstrip(".")
            )
    except ValueError:
        normalized = token
    return (
        normalized + "%"
        if is_percent
        else normalized
    )
def extract_numbers(text: str) -> list[str]:
    return [
        normalize_number(token)
        for token in NUMBER_PATTERN.findall(str(text))
    ]
def multiset_precision_recall_f1(
    reference_items: list[str],
    prediction_items: list[str],
) -> tuple[float, float, float]:
    reference_counter = Counter(reference_items)
    prediction_counter = Counter(prediction_items)
    overlap = sum(
        (
            reference_counter
            & prediction_counter
        ).values()
    )
    precision = (
        overlap / sum(prediction_counter.values())
        if prediction_counter
        else float(not reference_counter)
    )
    recall = (
        overlap / sum(reference_counter.values())
        if reference_counter
        else float(not prediction_counter)
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )
    return precision, recall, f1
def try_parse_json(text: str) -> dict[str, Any] | None:
    value = str(text).strip()
    if not (
        value.startswith("{")
        and value.endswith("}")
    ):
        return None
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None
def extract_scene_id(text: str) -> str | None:
    parsed = try_parse_json(text)
    if parsed is not None and "scene_id" in parsed:
        return str(parsed["scene_id"]).strip()
    match = SCENE_ID_PATTERN.search(str(text))
    return match.group(1).strip() if match else None
def extract_category(
    text: str,
    patterns: list[re.Pattern],
) -> str | None:
    for pattern in patterns:
        match = pattern.search(str(text))
        if match:
            return match.group(1).title()
    return None
numeric_precision = []
numeric_recall = []
numeric_f1 = []
numeric_exact_match = []
reference_json = []
prediction_json = []
json_valid = []
json_key_recall = []
json_value_accuracy = []
reference_scene_ids = []
prediction_scene_ids = []
scene_id_accuracy = []
reference_burden = []
prediction_burden = []
burden_accuracy = []
reference_disruption = []
prediction_disruption = []
disruption_accuracy = []
for reference, prediction in zip(
    references,
    predictions,
):
    ref_numbers = extract_numbers(reference)
    pred_numbers = extract_numbers(prediction)
    precision, recall, f1 = (
        multiset_precision_recall_f1(
            ref_numbers,
            pred_numbers,
        )
    )
    numeric_precision.append(precision)
    numeric_recall.append(recall)
    numeric_f1.append(f1)
    numeric_exact_match.append(
        float(Counter(ref_numbers) == Counter(pred_numbers))
    )
    ref_json = try_parse_json(reference)
    pred_json = try_parse_json(prediction)
    reference_json.append(ref_json)
    prediction_json.append(pred_json)
    structured_reference = ref_json is not None
    if structured_reference:
        json_valid.append(float(pred_json is not None))
        if pred_json is None:
            json_key_recall.append(0.0)
            json_value_accuracy.append(0.0)
        else:
            expected_keys = set(ref_json)
            predicted_keys = set(pred_json)
            matching_keys = expected_keys & predicted_keys
            json_key_recall.append(
                len(matching_keys) / len(expected_keys)
                if expected_keys
                else 1.0
            )
            correct_values = sum(
                str(pred_json[key]).strip().lower()
                == str(ref_json[key]).strip().lower()
                for key in matching_keys
            )
            json_value_accuracy.append(
                correct_values / len(expected_keys)
                if expected_keys
                else 1.0
            )
    else:
        json_valid.append(np.nan)
        json_key_recall.append(np.nan)
        json_value_accuracy.append(np.nan)
    ref_scene = extract_scene_id(reference)
    pred_scene = extract_scene_id(prediction)
    reference_scene_ids.append(ref_scene)
    prediction_scene_ids.append(pred_scene)
    scene_id_accuracy.append(
        float(
            ref_scene is not None
            and pred_scene is not None
            and ref_scene.lower() == pred_scene.lower()
        )
        if ref_scene is not None
        else np.nan
    )
    ref_burden = extract_category(
        reference,
        FLOOD_BURDEN_PATTERNS,
    )
    pred_burden = extract_category(
        prediction,
        FLOOD_BURDEN_PATTERNS,
    )
    reference_burden.append(ref_burden)
    prediction_burden.append(pred_burden)
    burden_accuracy.append(
        float(
            ref_burden is not None
            and pred_burden is not None
            and ref_burden == pred_burden
        )
        if ref_burden is not None
        else np.nan
    )
    ref_disruption = extract_category(
        reference,
        DISRUPTION_PATTERNS,
    )
    pred_disruption = extract_category(
        prediction,
        DISRUPTION_PATTERNS,
    )
    reference_disruption.append(ref_disruption)
    prediction_disruption.append(pred_disruption)
    disruption_accuracy.append(
        float(
            ref_disruption is not None
            and pred_disruption is not None
            and ref_disruption == pred_disruption
        )
        if ref_disruption is not None
        else np.nan
    )
metric_df["numeric_precision"] = numeric_precision
metric_df["numeric_recall"] = numeric_recall
metric_df["numeric_f1"] = numeric_f1
metric_df["numeric_exact_match"] = numeric_exact_match
metric_df["reference_scene_id_extracted"] = (
    reference_scene_ids
)
metric_df["prediction_scene_id_extracted"] = (
    prediction_scene_ids
)
metric_df["scene_id_accuracy"] = scene_id_accuracy
metric_df["reference_flood_burden"] = reference_burden
metric_df["prediction_flood_burden"] = prediction_burden
metric_df["flood_burden_accuracy"] = burden_accuracy
metric_df["reference_disruption"] = reference_disruption
metric_df["prediction_disruption"] = prediction_disruption
metric_df["disruption_accuracy"] = disruption_accuracy
metric_df["json_valid"] = json_valid
metric_df["json_key_recall"] = json_key_recall
metric_df["json_value_accuracy"] = json_value_accuracy
def safe_mean(series: pd.Series) -> float | None:
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()
    return (
        float(values.mean())
        if not values.empty
        else None
    )
extended_metrics = {
    "prediction_file": str(predictions_path),
    "test_records": int(len(metric_df)),
    "text_generation": {
        "bleu": float(bleu),        "rouge1_precision": safe_mean(
            metric_df["rouge1_precision"]
        ),
        "rouge1_recall": safe_mean(
            metric_df["rouge1_recall"]
        ),
        "rouge1_f1": safe_mean(
            metric_df["rouge1_f1"]
        ),
        "rouge2_precision": safe_mean(
            metric_df["rouge2_precision"]
        ),
        "rouge2_recall": safe_mean(
            metric_df["rouge2_recall"]
        ),
        "rouge2_f1": safe_mean(
            metric_df["rouge2_f1"]
        ),
        "rougeL_precision": safe_mean(
            metric_df["rougeL_precision"]
        ),
        "rougeL_recall": safe_mean(
            metric_df["rougeL_recall"]
        ),
        "rougeL_f1": safe_mean(
            metric_df["rougeL_f1"]
        ),
        "bertscore_precision": safe_mean(
            metric_df["bertscore_precision"]
        ),
        "bertscore_recall": safe_mean(
            metric_df["bertscore_recall"]
        ),
        "bertscore_f1": safe_mean(
            metric_df["bertscore_f1"]
        ),
    },
    "grounding_and_factuality": {
        "numeric_precision": safe_mean(
            metric_df["numeric_precision"]
        ),
        "numeric_recall": safe_mean(
            metric_df["numeric_recall"]
        ),
        "numeric_f1": safe_mean(
            metric_df["numeric_f1"]
        ),
        "numeric_exact_match": safe_mean(
            metric_df["numeric_exact_match"]
        ),
        "scene_id_accuracy": safe_mean(
            metric_df["scene_id_accuracy"]
        ),
        "flood_burden_accuracy": safe_mean(
            metric_df["flood_burden_accuracy"]
        ),
        "critical_disruption_accuracy": safe_mean(
            metric_df["disruption_accuracy"]
        ),
    },
    "structured_outputs": {
        "structured_reference_count": int(
            pd.Series(reference_json)
            .notna()
            .sum()
        ),
        "json_validity": safe_mean(
            metric_df["json_valid"]
        ),
        "json_key_recall": safe_mean(
            metric_df["json_key_recall"]
        ),
        "json_value_accuracy": safe_mean(
            metric_df["json_value_accuracy"]
        ),
    },
    "bertscore_configuration": {
        "model_type": bertscore_model,
        "device": bert_device,
    },
}
extended_csv = (
    metric_run_dir
    / "test_predictions_extended_metrics.csv"
)
extended_json = (
    metric_run_dir
    / "extended_evaluation_metrics.json"
)
metric_df.to_csv(
    extended_csv,
    index=False,
)
extended_json.write_text(
    json.dumps(
        extended_metrics,
        indent=2,
    ),
    encoding="utf-8",
)
print(
    json.dumps(
        extended_metrics,
        indent=2,
    )
)
